# Imports

In [5]:
from langchain_community.llms import LlamaCpp
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA

# Config

In [ ]:
GENERATION_MODEL_PATH = 'Phi-3-mini-4k-instruct-fp16.gguf'
RETREIVAL_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

In [34]:
FACTUAL_DATA = """
A science fiction epic released in 2016, titled 'Cosmos Voyage', was 
co-written and directed by Ethan Caldwell. The cast includes Liam 
McFarland, Emma Hartley, Julia Stanton, Ben Irwin, Ellen Barrett, Matt 
Lawson, and Michael Kane. The story is set in a bleak future where humanity 
is facing extinction. A team of astronauts embarks on a journey through a 
mysterious wormhole located near Jupiter, seeking a new planet capable of 
sustaining human life. The screenplay was created by Ethan and his brother 
Jonathan Caldwell, building on a story Jonathan had drafted in 2008. The 
project had scientific guidance from Dr. Alan Pierce, a theoretical 
physicist at MIT and 2018 Nobel laureate in Physics. He acted as an executive 
producer, offered scientific consultancy, and published a companion book, 
'The Science Behind Cosmos Voyage'. Cinematography was led by Hoyte Van 
Lytton, who shot on 35 mm Panavision film and 70 mm IMAX. Principal 
photography started in early 2015, with locations including Alberta, 
Greenland, and Los Angeles. The film used a combination of practical effects, 
miniatures, and digital visual effects created by the studio Digital Mirage. 
'Cosmos Voyage' premiered on November 12, 2016, initially on traditional 
film reels, before expanding to theaters with digital projection. It generated income 
over $712 million worldwide ($790 million including subsequent re-releases), 
making it one of the top-grossing films of 2016. Critics praised its 
performances, direction, screenplay, musical score, visual effects, thematic 
ambition, and emotional impact. Many astronomers also lauded its accurate 
depiction of astrophysical concepts. Since release, the movie has developed 
a dedicated fan following and is widely regarded as a standout in modern 
science fiction cinema. At the 89th Academy Awards, it received five 
nominations, winning Best Visual Effects, along with several other accolades."""

In [35]:
len(FACTUAL_DATA)

1939

# Download Model

In [36]:
!wget --no-clobber "https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf"

File ‘Phi-3-mini-4k-instruct-fp16.gguf’ already there; not retrieving.



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


# RAG System

## Generation Model

In [37]:
llm = LlamaCpp(
    model_path=GENERATION_MODEL_PATH,
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False,
)

## Retrieval Model

In [38]:
texts = FACTUAL_DATA.split('.')
texts = [t.strip(' \n') for t in texts]

In [49]:
embedding_model = HuggingFaceEmbeddings(model_name=RETREIVAL_MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 81899303-13dd-4b27-9a4c-758a41fee0d4)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/README.md
Retrying in 1s [Retry 1/5].


README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [50]:
retreival_db = FAISS.from_texts(texts, embedding_model)

## RAG Prompt

In [53]:
template = """<|user|>
Relevant information:
{context}
Provide a concise answer the following question using the relevant information
provided above:
{question}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"],
)

## RAG Pipeline

In [54]:
rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retreival_db.as_retriever(),
    chain_type_kwargs={
        "prompt": prompt
    },
    verbose=True,
)

## Invoke

In [55]:
rag.invoke('What is the income generated by Cosmos Voyage film?')



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'What is the income generated by Cosmos Voyage film?',
 'result': " The 'Cosmos Voyage' film generated an income of over $712 million worldwide, with $790 million including subsequent re-releases."}

In [30]:
rag.invoke('What is the income generated by God Father?')



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'What is the income generated by God Father?',
 'result': ' I\'m sorry for the confusion, but there seems to be an error. "Godfather" refers to The Godfather film series, particularly The Godfather (1972) and its sequel, The Godfather Part II (1974). Based on your provided details, it appears you might be referring to a different project with similar elements or perhaps another movie that earned over $712 million worldwide in 2016. However, using the information given, we can\'t accurately determine an income for "The Godfather" as the specified figures are related to later re-releases and do not pertain directly to the original films\' earnings at their time of release.\n\nIf you meant another movie with a similar name or involved Dr. Jonathan Caldwell, please provide additional details so I can assist you further. For accurate financial data on "The Godfather" (1972) and its sequel, one could refer to historical box office records which show that The Godfather earned over $

In [56]:
rag.invoke('Which awards has Cosmos Voyage won؟')



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Which awards has Cosmos Voyage won؟',
 'result': ' Cosmos Voyage won the Academy Award for Best Visual Effects at the 89th Academy Awards. It also received several other accolades, although specific additional awards are not listed in the provided information.'}

In [58]:
rag.invoke('Who are the actors in Cosmos Voyage?')



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Who are the actors in Cosmos Voyage?',
 'result': ' The actors in "Cosmos Voyage" are Liam McFarland, Emma Hartley, Julia Stanton, Ben Irwin, Ellen Barrett, Matt Lawson, and Michael Kane.'}